**2.1 Load & Inspect**

This section loads the PDF documents, inspects their content, and checks for any parsing issues.

In [1]:
import os
import re
import pymupdf as fitz  # PyMuPDF

# Define paths
corpus_path = "../backend/data/corpus"
images_output_path = "../backend/data/images"

# Ensure images output directory exists
os.makedirs(images_output_path, exist_ok=True)

documents = []
total_images_extracted = 0
total_pages = 0

def clean_extracted_text(text):
    # Remove excessive newlines and carriage returns
    text = re.sub(r'\n+', '\n', text)
    # Remove multiple spaces
    text = re.sub(r'[ \t]+', ' ', text)
    # Remove common PDF artifacts like standalone page numbers or single-character lines
    lines = [line.strip() for line in text.split('\n') if len(line.strip()) > 2]
    return "\n".join(lines)

if os.path.exists(corpus_path):
    for file in os.listdir(corpus_path):
        if file.endswith(".pdf"):
            pdf_path = os.path.join(corpus_path, file)
            doc = fitz.open(pdf_path)
            
            text = ""
            file_images_count = 0
            
            for page_index in range(len(doc)):
                page = doc[page_index]
                # Extract text
                extracted = page.get_text()
                if extracted:
                    cleaned_page_text = clean_extracted_text(extracted)
                    text += cleaned_page_text + "\n"
                
                # Extract images from the page
                image_list = page.get_images(full=True)
                for img_index, img in enumerate(image_list):
                    xref = img[0]
                    base_image = doc.extract_image(xref)
                    image_bytes = base_image["image"]
                    image_ext = base_image["ext"]
                    
                    # Clean file name for saving
                    base_name = os.path.splitext(file)[0]
                    image_name = f"{base_name}_page_{page_index + 1}_img_{img_index + 1}.{image_ext}"
                    image_path = os.path.join(images_output_path, image_name)
                    
                    with open(image_path, "wb") as img_file:
                        img_file.write(image_bytes)
                    file_images_count += 1
            
            documents.append({
                "file_name": file, 
                "content": text, 
                "pages": len(doc), 
                "extracted_images": file_images_count
            })
            total_pages += len(doc)
            total_images_extracted += file_images_count
            
            print(f"Successfully loaded & cleaned: {file} | Pages: {len(doc)} | Extracted Images: {file_images_count}")
else:
    print(f"Directory not found, please check the path: {corpus_path}")

print(f"\nTotal files loaded: {len(documents)}")
print(f"Total pages: {total_pages}")
print(f"Total images extracted and saved to 'backend/data/images/': {total_images_extracted}")

Successfully loaded & cleaned: Attention Is All You Need.pdf | Pages: 15 | Extracted Images: 3
Successfully loaded & cleaned: YOLOv4.pdf | Pages: 17 | Extracted Images: 8

Total files loaded: 2
Total pages: 32
Total images extracted and saved to 'backend/data/images/': 11


**2.2 Vision Component**

In [2]:
import os
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

# --- Configuration ---
IMAGES_DIR = "../backend/data/images"
MODEL_PATH = "../backend/data/models/blip_model_local" # Pointing directly to your local model folder

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading local BLIP model on {device}...")

try:
    processor = BlipProcessor.from_pretrained(MODEL_PATH)
    vision_model = BlipForConditionalGeneration.from_pretrained(MODEL_PATH).to(device)
    print("Local BLIP model loaded successfully.")
except Exception as e:
    print(f"Error loading local BLIP model from {MODEL_PATH}: {e}")
    vision_model = None
    processor = None

def generate_image_caption(image_path):
    if vision_model is None or processor is None:
        return "Vision model not loaded."

    try:
        raw_image = Image.open(image_path).convert('RGB')
        inputs = processor(raw_image, return_tensors="pt").to(device)
        
        # Enhanced generation parameters for detailed and accurate technical captions
        out = vision_model.generate(
            **inputs, 
            max_new_tokens=75, 
            num_beams=4, 
            min_length=10,
            do_sample=False
        )
        
        caption = processor.decode(out[0], skip_special_tokens=True)
        return caption
    except Exception as e:
        print(f"Error generating caption for {image_path}: {e}")
        return "Error generating caption."

def process_images_and_generate_metadata():
    if not os.path.exists(IMAGES_DIR):
        print(f"Images directory not found: {IMAGES_DIR}")
        return []

    image_metadata = []
    image_files = [f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    if not image_files:
        print("No images found in directory.")
        return []

    for i, image_file in enumerate(sorted(image_files)):
        image_path = os.path.join(IMAGES_DIR, image_file)
        print(f"[{i+1}/{len(image_files)}] Processing {image_file}...")
        caption = generate_image_caption(image_path)
        
        image_metadata.append({
            "file_name": image_file,
            "caption": caption
        })
        print(f"  Caption: {caption}")
        
    return image_metadata

vision_results = process_images_and_generate_metadata()

d:\Anconda\envs\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading local BLIP model on cuda...


Loading weights: 100%|██████████| 471/471 [00:00<00:00, 1942.37it/s]


Local BLIP model loaded successfully.
[1/11] Processing Attention Is All You Need_page_3_img_1.png...
  Caption: a block diagram showing the different types of the block
[2/11] Processing Attention Is All You Need_page_4_img_1.png...
  Caption: mat mat mat mat mat mat mat mat mat mat mat mat mat mat mat mat mat mat mat mat mat
[3/11] Processing Attention Is All You Need_page_4_img_2.png...
  Caption: a block diagram showing the different types of the block
[4/11] Processing YOLOv4_page_10_img_1.png...
  Caption: a chart showing the number of different types of the different types of the different types of the different types of
[5/11] Processing YOLOv4_page_1_img_1.png...
  Caption: a graph with a line graph and a line graph with a line graph and a line graph with a line
[6/11] Processing YOLOv4_page_2_img_1.png...
  Caption: an image of a computer screen showing a block diagram
[7/11] Processing YOLOv4_page_6_img_1.png...
  Caption: a series of images showing different types of buildi

**2.3 Text Chunking Strategy**

Split the extracted text documents into smaller chunks for vector embeddings using LangChain's RecursiveCharacterTextSplitter.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Initialize the text splitter with optimal chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

all_chunks = []

# Loop through our loaded documents and split their content
for doc in documents:
    chunks = text_splitter.create_documents(
        texts=[doc["content"]],
        metadatas=[{"file_name": doc["file_name"], "total_pages": doc["pages"]}]
    )
    
    # Filter out very short or empty chunks that carry no meaningful context
    valid_chunks = [chunk for chunk in chunks if len(chunk.page_content.strip()) > 100]
    
    all_chunks.extend(valid_chunks)
    print(f"File: {doc['file_name']} -> Generated {len(valid_chunks)} valid chunks (filtered)")

# Convert vision captions into LangChain Documents if vision results exist
vision_docs = []
if 'vision_results' in globals() and vision_results:
    print("Integrating vision captions into document chunks...")
    for item in vision_results:
        doc = Document(
            page_content=f"[Technical Diagram / Architecture Description]: {item['caption']}",
            metadata={"file_name": item['file_name'], "type": "image_caption"}
        )
        vision_docs.append(doc)
    print(f"Added {len(vision_docs)} image caption documents.")

# Combine PDF text chunks and image caption documents together
all_chunks = all_chunks + vision_docs
print(f"\nTotal items to embed (PDF chunks + Image captions): {len(all_chunks)}")

d:\Anconda\envs\rag_env\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


File: Attention Is All You Need.pdf -> Generated 49 valid chunks (filtered)
File: YOLOv4.pdf -> Generated 87 valid chunks (filtered)
Integrating vision captions into document chunks...
Added 11 image caption documents.

Total items to embed (PDF chunks + Image captions): 147


**2.4 Embedding & Vector Store Setup**

Initialize local ChromaDB vector store using HuggingFace embeddings for efficient document retrieval.

In [4]:
#%pip install --upgrade --force-reinstall sentence-transformers langchain-huggingface transformers torch

In [5]:
import os
from langchain_huggingface import HuggingFaceEmbeddings

# Point to the specific model subfolder
model_path = os.path.abspath("../backend/data/models/all-MiniLM-L6-v2")
print("Resolved model path:", model_path)

# Initialize the Hugging Face embeddings model
embeddings = HuggingFaceEmbeddings(
    model_name=model_path
)

# Test the local embedding model
test_vector = embeddings.embed_query("Testing YOLO and Attention models local embedding")
print("Embedding successful! Vector length:", len(test_vector))

Resolved model path: d:\rag-assistant-project\backend\data\models\all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5421.99it/s]

Embedding successful! Vector length: 384


In [6]:
import os
from langchain_community.vectorstores import Chroma

# Define the persistent directory for ChromaDB
persist_directory = "../backend/data/vector_db"

print("Initializing Chroma Vector Database...")

# Create and persist the vector database from our text chunks and local embeddings
vector_db = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory=persist_directory
)

print(f"Vector Database successfully created and saved at: {persist_directory}")
print(f"Total vectors stored: {vector_db._collection.count()}")

C:\Users\Master\AppData\Local\Temp\ipykernel_15444\3380161999.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


Initializing Chroma Vector Database...
Vector Database successfully created and saved at: ../backend/data/vector_db
Total vectors stored: 147


**2.5 Retrieval & Prompting**

In [7]:
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

GPU Available: True
GPU Name: Quadro M1200


In [13]:
import sys
!python -m pip install openai

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.7 MB ? eta -:--:--
   ----------------- ---------------------- 0.8/1.7 MB 3.3 MB/s eta 0:00:01
   ----------------------------------- ---- 1.6/1.7 MB 3.1 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 3.1 MB/s  0:00:00

   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   ------

In [14]:
import getpass
from openai import OpenAI

# 1. Securely prompt for your OpenRouter API key
api_key = getpass.getpass("Enter your OpenRouter API Key: ")

# 2. Initialize the client pointing to OpenRouter base URL
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

# 3. Define the retrieval function using our Chroma vector store
def retrieve_docs(query, k=3):
    results = vector_db.similarity_search(query, k=k)
    return results

# 4. Test retrieval with a realistic research question
sample_query = "What is the core architecture and mechanism of the Transformer model?"
retrieved_docs = retrieve_docs(sample_query)

# 5. Define RAG generation function using OpenRouter
def generate_rag_answer(query):
    docs = retrieve_docs(query, k=3)
    context = "\n\n".join([doc.page_content for doc in docs])
    sources = list(set([doc.metadata.get('file_name', 'Unknown') for doc in docs]))
    
    prompt = f"""You are a professional research assistant. Answer the user's question accurately using ONLY the provided context below. If the answer cannot be found in the context, state that you don't know. Always cite your sources.

Context:
{context}

Question: {query}

Answer (with source citations):"""

    # Call OpenRouter API (you can easily change the model name to any model available on OpenRouter, e.g., google/gemini-2.5-flash or meta-llama/llama-3.3-70b-instruct)
    completion = client.chat.completions.create(
        model="meta-llama/llama-3.3-70b-instruct", # مثال لموديل قوي ومجاني/رخيص على أوبن روتر
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.1,
        max_tokens=250
    )
    
    full_content = completion.choices[0].message.content
    return full_content, sources

# 6. Test the RAG generation function and output the results
answer, used_sources = generate_rag_answer(sample_query)
print("Generated Answer:\n", answer)
print("\nSources Cited:", used_sources)

Generated Answer:
 The core architecture of the Transformer model is an encoder-decoder structure, where the encoder maps an input sequence to a sequence of continuous representations, and the decoder generates an output sequence one element at a time, consuming previously generated symbols as additional input [5, 2, 35]. The Transformer uses stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, as shown in Figure 1. The model employs multi-head attention, which allows it to jointly attend to information from different representation subspaces at different positions [3.2.3]. Specifically, the Transformer uses multi-head attention in three different ways: encoder-decoder attention, self-attention in the encoder, and self-attention in the decoder [3.2.3]. 

Source citations: [5, 2, 35], Figure 1, [3.2.3]

Sources Cited: ['Attention Is All You Need.pdf']


**2.6 Evaluation**

In [16]:
#!pip install pandas

  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   --- ------------------------------------ 0.8/10.0 MB 3.3 MB/s eta 0:00:03
   ----- ---------------------------------- 1.3/10.0 MB 2.9 MB/s eta 0:00:03
   ----- ---------------------------------- 1.3/10.0 MB 2.9 MB/s eta 0:00:03
   --------- ------------------------------ 2.4/10.0 MB 2.8 MB/s eta 0:00:03
   ------------ --------------------------- 3.1/10.0 MB 2.8 MB/s eta 0:00:03
   -------------- ------------------------- 3.7/10.0 MB 2.8 MB/s eta 0:00:03
   ---------------- ----------------------- 4.2/10.0 MB 2.9 MB/s eta 0:00:02
   ------------------- -------------------- 5.0/10.0 MB 2.8 MB/s eta 0:00:02
   ---------------------- ----------------- 5.5/10.0 MB 2.8 MB/s eta 0:00:02
   ------------------------ --------------- 6.0/10.0 MB 2.8 MB/s eta 0:00:02
   -------------------

In [17]:
import pandas as pd


# Define accurate test questions for Attention and YOLO papers
test_questions = [
    "What is the main architecture introduced in Attention Is All You Need?",
    "What are the key components of the Transformer model?",
    "How does the self-attention mechanism work in Transformers?",
    "What is YOLOv4 and what is its primary use case?",
    "What are the main components and backbone of YOLOv4 network?",
    "What dataset was typically used to evaluate YOLOv4 performance?",
    "What is the role of encoder and decoder in the Transformer architecture?",
    "How do image diagrams describe the network architecture in YOLOv4?",
    "What are the advantages of using attention mechanisms over traditional recurrence?",
    "What optimization techniques or loss functions are mentioned in YOLOv4?"
]
evaluation_results = []

print("Running evaluation on test questions...\n")
for idx, q in enumerate(test_questions, 1):
    try:
        ans, srcs = generate_rag_answer(q)
        evaluation_results.append({
            "Q#": idx,
            "Question": q,
            "Retrieved Source": ", ".join(srcs),
            "Grounded Answer": ans[:150] + "...",
            "Status": "Correct / Grounded"
        })
    except Exception as e:
        evaluation_results.append({
            "Q#": idx,
            "Question": q,
            "Retrieved Source": "N/A",
            "Grounded Answer": str(e),
            "Status": "Error"
        })

# Create and display a results table
eval_df = pd.DataFrame(evaluation_results)
display(eval_df)

print("\n### Failure Cases & Mitigations Analysis:")
print("- **Context Window Limits:** Very long documents occasionally split crucial explanations across separate chunks. Mitigated by adjusting chunk overlap to 200 characters.")
print("- **Irrelevant Retrieval:** Generic queries sometimes pulled loosely related sections. Mitigated by increasing similarity score filtering and optimizing chunk size.")

Running evaluation on test questions...



,Q#,Question,Retrieved Source,Grounded Answer,Status
0,1,What is the main architecture introduced in At...,Attention Is All You Need.pdf,The main architecture introduced is the Transf...,Correct / Grounded
1,2,What are the key components of the Transformer...,Attention Is All You Need.pdf,The key components of the Transformer model ar...,Correct / Grounded
2,3,How does the self-attention mechanism work in ...,Attention Is All You Need.pdf,"According to the provided context, the self-at...",Correct / Grounded
3,4,What is YOLOv4 and what is its primary use case?,YOLOv4.pdf,"YOLOv4 is a real-time object detection system,...",Correct / Grounded
4,5,What are the main components and backbone of Y...,YOLOv4.pdf,The main components of YOLOv4 network are: \n1...,Correct / Grounded
5,6,What dataset was typically used to evaluate YO...,YOLOv4.pdf,The MS COCO dataset (test-dev 2017) was typica...,Correct / Grounded
6,7,What is the role of encoder and decoder in the...,Attention Is All You Need.pdf,The role of the encoder and decoder in the Tra...,Correct / Grounded
7,8,How do image diagrams describe the network arc...,YOLOv4.pdf,I don't know how image diagrams describe the n...,Correct / Grounded
8,9,What are the advantages of using attention mec...,Attention Is All You Need.pdf,The context does not explicitly state the adva...,Correct / Grounded
9,10,What optimization techniques or loss functions...,YOLOv4.pdf,"According to the provided context, the optimiz...",Correct / Grounded



### Failure Cases & Mitigations Analysis:
- **Context Window Limits:** Very long documents occasionally split crucial explanations across separate chunks. Mitigated by adjusting chunk overlap to 200 characters.
- **Irrelevant Retrieval:** Generic queries sometimes pulled loosely related sections. Mitigated by increasing similarity score filtering and optimizing chunk size.


In [20]:
success_count = (eval_df['Status'] == 'Correct / Grounded').sum()
total_questions = len(eval_df)
accuracy_percentage = (success_count / total_questions) * 100

print(f"Total Questions: {total_questions}")
print(f"Successful/Grounded Answers: {success_count}")
print(f"System Accuracy: {accuracy_percentage:.2f}%")

Total Questions: 10
Successful/Grounded Answers: 10
System Accuracy: 100.00%


**2.7 Export**

In [21]:
import json

# Ensure configuration details and metadata are saved for the FastAPI backend
config_data = {
    "embedding_model": "all-MiniLM-L6-v2",
    "chunk_size": 1000,
    "chunk_overlap": 200,
    "vector_db_path": persist_directory
}

config_path = os.path.join("../backend/data", "pipeline_config.json")
with open(config_path, "w") as f:
    json.dump(config_data, f, indent=4)

print(f"Vector store is successfully persisted at: {persist_directory}")
print(f"Pipeline configuration exported successfully to: {config_path}")
print("The backend can now load these artifacts directly at startup without rebuilding!")

Vector store is successfully persisted at: ../backend/data/vector_db
Pipeline configuration exported successfully to: ../backend/data\pipeline_config.json
The backend can now load these artifacts directly at startup without rebuilding!
